# **Employee Attendance in Bangalore Data Cleaning**

**Dataset:** `Employee-attendance-and-login-logout-data-bangalore.csv` (55,374 rows × 22 columns) <br/>

This notebook walks through **all 23 steps** of the checklist, in order, across four phases:

| Phase | Steps | Goal |
|---|---|---|
| Phase 1 — Inspect | 1–8 | Understand the shape, structure, and quality of the raw data |
| Phase 2 — Clean & Prepare | 9–17 | Organize columns, clean values, export a final clean dataset |

Each step below has its own markdown explanation followed by the code that performs it.

## **Environment Setup :**

In [2]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

**Load the Dataset :**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
RAW_PATH = "/content/drive/MyDrive/employee_attendance_bangalore_q1_2026.csv"
df = pd.read_csv(RAW_PATH)
df.head()

,attendance_id,employee_id,employee_name,gender,department,designation,employment_type,office_location,date_of_joining,attendance_date,shift_type,attendance_status,work_mode,login_timestamp,logout_timestamp,total_hours_worked,break_duration_mins,net_productive_hours,late_arrival_mins,early_exit_mins,overtime_hours,leave_type
0,ATT0000001,VL1000,Arjun Verma,Male,Engineering,Software Engineer Intern,Intern,Koramangala HQ,2025-06-26,2026-01-02,General (09:30-18:30),Present,Work From Office,2026-01-02 09:42:03,2026-01-02 18:34:12,8.87,86,7.44,12,0,0.00,Not Applicable
1,ATT0000002,VL1001,Rekha Chatterjee,Female,Operations,Operations Manager,Full-Time,Koramangala HQ,2024-02-16,2026-01-02,General (09:30-18:30),Present,Work From Home,2026-01-02 09:09:11,2026-01-02 18:25:11,9.27,67,8.15,0,6,0.00,Not Applicable
2,ATT0000003,VL1002,Tanmay Bhardwaj,Male,Operations,Operations Analyst,Full-Time,Whitefield Tech Park,2020-10-13,2026-01-02,General (09:30-18:30),Present,Work From Home,2026-01-02 09:45:59,2026-01-02 20:17:31,10.53,57,9.58,15,0,0.58,Not Applicable
3,ATT0000004,VL1003,Aditya Bose,Female,Customer Success,Customer Success Executive,Full-Time,Koramangala HQ,2022-01-10,2026-01-02,Late/US Overlap (14:00-23:00),Present,Work From Office,2026-01-02 13:49:26,2026-01-02 22:24:51,8.59,76,7.32,0,36,0.00,Not Applicable
4,ATT0000005,VL1004,Uday Acharya,Male,Human Resources,Talent Acquisition Partner,Full-Time,Electronic City Campus,2024-09-09,2026-01-02,Flexible (10:30-19:30),Present,Work From Office,2026-01-02 10:51:02,2026-01-02 20:33:39,9.71,61,8.69,20,0,0.00,Not Applicable


## Phase 1 — Inspect the Raw Data (Steps 1–8)

Understand the shape, structure, and quality of the raw data.

### Step 1 — Total Number of Rows
Check the row count to understand the dataset's size (`df.shape[0]`).

In [5]:
n_rows = df.shape[0]
print(f"Total rows: {n_rows}")

Total rows: 55374


### Step 2 — Total Number of Columns
Check the column count (`df.shape[1]`).

In [6]:
n_cols = df.shape[1]
print(f"Total columns: {n_cols}")

Total columns: 22


### Step 3 — Understanding of Each Column
Go through every column and note what it represents, its expected values, and how it relates to the problem.

| Column | Represents | Expected values |
|---|---|---|
| attendance_id | Unique identifier for each daily attendance entry | `ATTxxxxxxx` (e.g., `ATT0000001` to `ATT0055374`) |
| employee_id | Unique identifier for each employee | `VLxxxx` (980 unique IDs, e.g., `VL1000` to `VL1979`) |
| employee_name | Full name of the employee | Free text (e.g., Arjun Verma, Rekha Chatterjee) |
| gender | Gender identity of the employee | Male, Female, Prefer Not to Say |
| department | Business department/unit | 10 unique departments (Engineering, Sales, Operations, Data Science, etc.) |
| designation | Job role or title | 48 unique titles (e.g., Software Engineer, Operations Analyst) |
| employment_type | Employment contract type | Full-Time, Contract, Intern, Part-Time |
| office_location | Assigned physical office campus | 5 Bangalore locations (Koramangala HQ, Whitefield Tech Park, HSR Layout Hub, Indiranagar Annexe, Electronic City Campus) |
| date_of_joining | Official employment start date | Date string format `YYYY-MM-DD` |
| attendance_date | Date of the attendance record | Date string format `YYYY-MM-DD` (57 working days in Q1 2026) |
| shift_type | Scheduled work shift and hours | 5 shift types (e.g., General (09:30-18:30), Flexible (10:30-19:30), Mid (11:00-20:00), Late/US Overlap (14:00-23:00), Early (08:00-17:00)) |
| attendance_status | Operational status for the day | Present, Half Day, On Leave |
| work_mode | Work location status | Work From Office, Work From Home, Client Site, Not Applicable (when on leave) |
| login_timestamp | System punch-in date and time | Timestamp `YYYY-MM-DD HH:MM:SS`, blank for leave days |
| logout_timestamp | System punch-out date and time | Timestamp `YYYY-MM-DD HH:MM:SS`, blank for leave days |
| total_hours_worked | Calculated gross duration between login and logout | `0.00` to `13.66` hours (`0.0` on leave days) |
| break_duration_mins | Duration spent on official breaks during shift | `0` to `135` minutes |
| net_productive_hours | Active work duration (`total_hours_worked` minus `break_duration_mins`) | `0.00` to `12.86` hours |
| late_arrival_mins | Delay in login past shift start time | `0` to `218` minutes |
| early_exit_mins | Departure time before shift end time | `0` to `391` minutes |
| overtime_hours | Hours worked beyond standard shift duration | `0.00` to `3.86` hours |
| leave_type | Classification of leave taken | Not Applicable, Sick Leave, Casual Leave, Earned Leave, Loss of Pay, Bereavement Leave, Marriage Leave, Work From Home Comp-Off, Parental Leave |

In [7]:
# Create a summary DataFrame describing structure of df
info_df = pd.DataFrame({

    # Data type of each column
    "dtype": df.dtypes.astype(str),

    # Number of unique values per column
    "n_unique": df.nunique(),

    # First-row sample value for each column
    "sample_value": df.iloc[0]
})

info_df

,dtype,n_unique,sample_value
attendance_id,object,55374,ATT0000001
employee_id,object,980,VL1000
employee_name,object,980,Arjun Verma
gender,object,3,Male
department,object,10,Engineering
designation,object,48,Software Engineer Intern
employment_type,object,4,Intern
office_location,object,5,Koramangala HQ
date_of_joining,object,804,2025-06-26
attendance_date,object,57,2026-01-02


### Step 4 — Trim Extra Spaces
Strip leading/trailing whitespace from string columns and column headers — hidden spaces silently
break groupby, filtering, and joins. We first **detect** which columns are affected before fixing them
(the actual fix happens formally in Step 16, but we flag it here as required by Step 4).

In [8]:
print("Header whitespace issues:", [c for c in df.columns if c != c.strip()])

obj_cols = df.select_dtypes(include="object").columns
whitespace_flagged = {}
for col in obj_cols:
    s = df[col].dropna().astype(str)
    whitespace_flagged[col] = int((s.str.strip() != s).sum())

flagged = {k: v for k, v in whitespace_flagged.items() if v > 0}
print(flagged if flagged else "No leading/trailing whitespace found in any string column.")

Header whitespace issues: []
No leading/trailing whitespace found in any string column.


### Step 5 — Remove Duplicate Elements
Identify and drop duplicate rows (`df.duplicated()`, `df.drop_duplicates()`).

In [9]:
n_dupes_full = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes_full}")
n_dupes_id = df["attendance_id"].duplicated().sum()
print(f"Duplicated attendance_id values: {n_dupes_id}")

Fully duplicated rows: 0
Duplicated attendance_id values: 0


### Step 6 — Check Memory Size
Check memory usage (`df.memory_usage(deep=True)`) — flags if dtypes need downcasting for efficiency.

In [10]:
mem = df.memory_usage(deep=True)
print(mem)
print(f"\nTotal memory: {mem.sum() / 1024**2:.2f} MB")

Index                       132
attendance_id           3267066
employee_id             3045570
employee_name           3442982
gender                  2980742
department              3312109
designation             3830839
employment_type         3198638
office_location         3651166
date_of_joining         3267066
attendance_date         3267066
shift_type              3904830
attendance_status       3104765
work_mode               3564478
login_timestamp         3670572
logout_timestamp        3670572
total_hours_worked       442992
break_duration_mins      442992
net_productive_hours     442992
late_arrival_mins        442992
early_exit_mins          442992
overtime_hours           442992
leave_type              3483852
dtype: int64

Total memory: 54.66 MB


### Step 7 — Check Data Type of Columns
Verify each column's dtype matches what it should be (e.g., dates not stored as text, numbers not
stored as objects).

In [11]:
df.dtypes

,0
attendance_id,object
employee_id,object
employee_name,object
gender,object
department,object
designation,object
employment_type,object
office_location,object
date_of_joining,object
attendance_date,object


In [12]:
should_be_datetime = ["date_of_joining", "attendance_date", "login_timestamp", "logout_timestamp"]
print("Currently wrong dtype (datetime expected):")
print(df[should_be_datetime].dtypes)
print("\nNumeric columns already correctly typed:")
print(df.select_dtypes(include="number").dtypes)

Currently wrong dtype (datetime expected):
date_of_joining     object
attendance_date     object
login_timestamp     object
logout_timestamp    object
dtype: object

Numeric columns already correctly typed:
total_hours_worked      float64
break_duration_mins       int64
net_productive_hours    float64
late_arrival_mins         int64
early_exit_mins           int64
overtime_hours          float64
dtype: object


### Step 8 — Check Total Number of Null Values
Count missing values per column (`df.isnull().sum()`) to plan the cleaning strategy.

In [13]:
null_counts = df.isnull().sum().sort_values(ascending=False)
null_pct = (null_counts / len(df) * 100).round(2)
pd.DataFrame({"nulls": null_counts, "pct_missing": null_pct}).loc[null_counts > 0]

,nulls,pct_missing
logout_timestamp,2635,4.76
login_timestamp,2635,4.76


In [14]:
# Are the nulls explained by attendance_status? (login/logout should be null exactly on leave days)
print(pd.crosstab(df["attendance_status"], df["login_timestamp"].isnull()))
print("\n-> Nulls in login/logout_timestamp are fully explained by 'On Leave' status -- not a data")
print("   quality defect, just a structurally-absent value that should stay NaN.")

login_timestamp    False  True 
attendance_status              
Half Day            1186      0
On Leave               0   2635
Present            51553      0

-> Nulls in login/logout_timestamp are fully explained by 'On Leave' status -- not a data
   quality defect, just a structurally-absent value that should stay NaN.


## Phase 2 — Clean & Prepare (Steps 9–17)

Organize columns, clean values, and export a final clean dataset.

### Step 9 — Split Columns: Numerical vs Categorical
Separate columns into numerical and categorical groups — they need different analysis and cleaning approaches.

In [15]:
numerical_cols = ["total_hours_worked", "break_duration_mins", "net_productive_hours",
                   "late_arrival_mins", "early_exit_mins", "overtime_hours"]
categorical_cols = ["gender", "department", "designation", "employment_type", "office_location",
                     "shift_type", "attendance_status", "work_mode", "leave_type"]
datetime_cols = ["date_of_joining", "attendance_date", "login_timestamp", "logout_timestamp"]
id_cols = ["attendance_id", "employee_id"]

print("Numerical:", numerical_cols)
print("\nCategorical:", categorical_cols)
print("\nDate/Time:", datetime_cols)
print("\nID columns:", id_cols)

Numerical: ['total_hours_worked', 'break_duration_mins', 'net_productive_hours', 'late_arrival_mins', 'early_exit_mins', 'overtime_hours']

Categorical: ['gender', 'department', 'designation', 'employment_type', 'office_location', 'shift_type', 'attendance_status', 'work_mode', 'leave_type']

Date/Time: ['date_of_joining', 'attendance_date', 'login_timestamp', 'logout_timestamp']

ID columns: ['attendance_id', 'employee_id']


### Step 10 — describe() of Numerical Columns
Run df.describe() to see count, mean, std, min, quartiles, and max for numerical columns.*italicised text*

In [16]:
df[numerical_cols].describe()

,total_hours_worked,break_duration_mins,net_productive_hours,late_arrival_mins,early_exit_mins,overtime_hours
count,55374.000000,55374.000000,55374.000000,55374.000000,55374.000000,55374.000000
mean,8.724951,59.741955,7.729238,9.858471,20.099198,0.150049
std,2.413434,23.878616,2.223308,19.465159,49.630750,0.367258
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.470000,47.000000,7.370000,0.000000,0.000000,0.000000
50%,9.280000,62.000000,8.210000,0.000000,0.000000,0.000000
75%,10.000000,76.000000,8.960000,14.000000,16.000000,0.000000
max,13.660000,135.000000,12.860000,218.000000,391.000000,3.860000


### Step 11 — Write a Summary of the Dataset
**Summary:** This dataset is a daily attendance log for Vagu Labs' Bangalore workforce across Q1 2026 (Jan 2 – Mar 31). Each of the 55,374 rows is one employee-day record, covering 980 unique employees across 10 departments and 5 Bangalore office locations. It captures shift assignment, attendance outcome (Present / Half Day / On Leave), work mode (office / home / client site), clock-in/out timestamps, hours worked, breaks, punctuality (late arrival / early exit), overtime, and leave type.

**Source:** HR attendance export (`employee_attendance_bangalore_q1_2026.csv`), treated as a day-to-day operational dataset rather than a teaching-messy one.

**Size:** 55,374 rows × 22 columns, ~980 employees averaging ~57 records each (working days in Q1 2026 minus weekends).

**General quality:** Notably clean — zero duplicate rows, zero whitespace issues, zero inconsistent category labels, and employee-level attributes (`name, gender, department, designation, etc.`) are fully consistent across every row for the same employee_id. The only missing values (`login_timestamp / logout_timestamp`, 2,635 rows each) are structurally correct — they occur exactly on "On Leave" days, not as a data-entry gap. The main cleaning work needed is type correction (dates/timestamps stored as text) rather than fixing broken values.

In [17]:
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"Unique employees: {df['employee_id'].nunique()}")
print(f"Departments: {sorted(df['department'].unique())}")
attendance_date_notnull = df['attendance_date'].dropna()
print(f"Attendance date range: {attendance_date_notnull.min()} .. {attendance_date_notnull.max()}")
print(f"\nAttendance status breakdown:\n{df['attendance_status'].value_counts()}")

Rows: 55,374  |  Columns: 22
Unique employees: 980
Departments: ['Customer Success', 'Data Science', 'Design', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Product Management', 'Sales']
Attendance date range: 2026-01-02 .. 2026-03-31

Attendance status breakdown:
attendance_status
Present     51553
On Leave     2635
Half Day     1186
Name: count, dtype: int64


### Step 12 — Write the Problem Statement
**Problem Statement:** *What are the attendance, punctuality, and productivity patterns of Vagu Labs' Bangalore workforce in Q1 2026, and how do they vary by department, work mode, shift type, and employment type? Specifically, what is associated with late arrivals, early exits, and leave-taking — and is remote work adoption uneven across departments?*

This drives every scoping decision from here on — which columns are required, which derived metrics matter, and which relationships we test statistically.

### Step 13 — Mention the Columns That Are Required
Columns needed to answer the problem statement above:

*   **Identifiers (for grouping):** `employee_id`
*   **Who:** `gender, department, designation, employment_type, office_location, date_of_joining`

*   **When:** `attendance_date, shift_type`
*   **Outcome:** `attendance_status, work_mode, login_timestamp, logout_timestamp, leave_type`

*   **Effort/Punctuality:** `total_hours_worked, break_duration_mins, net_productive_hours, late_arrival_mins, early_exit_mins, overtime_hours`








### Step 14 — Drop the Columns That Are Not Required
Not needed to answer the problem statement:


*   `employee_name` — PII, not needed for aggregate/departmental analysis (employee_id retained for grouping)
*   `attendance_id` — pure row identifier with no analytic value once the DataFrame index exists



In [18]:
cols_to_drop = ["employee_name", "attendance_id"]
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} column(s). Remaining: {df.shape[1]}")
df.columns.tolist()

Dropped 2 column(s). Remaining: 20


['employee_id',
 'gender',
 'department',
 'designation',
 'employment_type',
 'office_location',
 'date_of_joining',
 'attendance_date',
 'shift_type',
 'attendance_status',
 'work_mode',
 'login_timestamp',
 'logout_timestamp',
 'total_hours_worked',
 'break_duration_mins',
 'net_productive_hours',
 'late_arrival_mins',
 'early_exit_mins',
 'overtime_hours',
 'leave_type']

### Step 15 — Add Derived Columns If Required
Useful columns not present in the raw data, created from existing ones:


*   `attendance_month, attendance_weekday` — extracted from `attendance_date` once parsed

*   `tenure_days` — `attendance_date` minus date_of_joining

*   `is_late` — `late_arrival_mins` > 0
*   `is_early_exit` — `early_exit_mins` > 0


*   `productivity_ratio` — `net_productive_hours / total_hours_worked` (only meaningful on worked days)


(Computed together with Step 16 below, since they depend on the parsed/typed versions of their source columns.)



### Step 16 — Perform Cleaning Operations on Each Column
Applying the right cleaning method based on each column's type and role: text/categorical standardization, numerical outlier/skew checks, date/time parsing, ID validation, and cross-column consistency checks.

**16a — Text / Categorical columns:** strip whitespace and check for inconsistent labels. As found in Step 4, this dataset has none — the step is still run (rather than skipped) so the notebook's cleaning logic is complete and reusable on a messier extract of the same data.

In [19]:
df.columns = df.columns.str.strip()
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()
    df.loc[df[c].isin(["", "nan", "None"]), c] = np.nan

for c in ["gender", "department", "employment_type", "work_mode", "attendance_status", "leave_type"]:
    print(f"{c}: {sorted(df[c].dropna().unique())}")

gender: ['Female', 'Male', 'Prefer Not to Say']
department: ['Customer Success', 'Data Science', 'Design', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Product Management', 'Sales']
employment_type: ['Contract', 'Full-Time', 'Intern', 'Part-Time']
work_mode: ['Client Site', 'Not Applicable', 'Work From Home', 'Work From Office']
attendance_status: ['Half Day', 'On Leave', 'Present']
leave_type: ['Bereavement Leave', 'Casual Leave', 'Earned Leave', 'Loss of Pay', 'Marriage Leave', 'Not Applicable', 'Parental Leave', 'Sick Leave', 'Work From Home Comp-Off']


**16b — Numerical columns:** outlier detection (IQR method) and skewness/kurtosis checks.

In [20]:
# Outlier detection on late_arrival_mins via IQR method
q1, q3 = df["late_arrival_mins"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["late_arrival_mins"] < lower) | (df["late_arrival_mins"] > upper)]
print(f"late_arrival_mins IQR bounds: [{max(lower,0):.0f}, {upper:.0f}]  |  Outliers flagged: {len(outliers)}")
print("These represent genuinely very-late arrivals, not data-entry errors -> kept as-is for analysis.")

late_arrival_mins IQR bounds: [0, 35]  |  Outliers flagged: 2899
These represent genuinely very-late arrivals, not data-entry errors -> kept as-is for analysis.


In [21]:
for c in numerical_cols:
    print(f"{c:22s} skew={df[c].skew():+.2f}   kurtosis={df[c].kurt():+.2f}")
print("\nlate_arrival_mins, early_exit_mins, and overtime_hours are strongly right-skewed, as")
print("expected: most days have 0 minutes late / 0 early exit / 0 overtime, with a long tail.")

total_hours_worked     skew=-2.38   kurtosis=+5.81
break_duration_mins    skew=-0.56   kurtosis=+0.35
net_productive_hours   skew=-2.12   kurtosis=+4.77
late_arrival_mins      skew=+4.20   kurtosis=+22.95
early_exit_mins        skew=+3.78   kurtosis=+16.00
overtime_hours         skew=+3.09   kurtosis=+10.85

late_arrival_mins, early_exit_mins, and overtime_hours are strongly right-skewed, as
expected: most days have 0 minutes late / 0 early exit / 0 overtime, with a long tail.


**16c — Date / Time columns:** parse `date_of_joining, attendance_date, login_timestamp, logout_timestamp` into proper datetime types, then extract derived parts.

In [22]:
df["date_of_joining"] = pd.to_datetime(df["date_of_joining"], errors="coerce")
df["attendance_date"] = pd.to_datetime(df["attendance_date"], errors="coerce")
df["login_timestamp"] = pd.to_datetime(df["login_timestamp"], errors="coerce")
df["logout_timestamp"] = pd.to_datetime(df["logout_timestamp"], errors="coerce")

print("Unparseable dates:")
print(df[["date_of_joining", "attendance_date", "login_timestamp", "logout_timestamp"]].isna().sum())

df["attendance_month"] = df["attendance_date"].dt.month
df["attendance_weekday"] = df["attendance_date"].dt.day_name()
df["tenure_days"] = (df["attendance_date"] - df["date_of_joining"]).dt.days
df[["attendance_date", "attendance_month", "attendance_weekday", "tenure_days"]].head()

Unparseable dates:
date_of_joining        0
attendance_date        0
login_timestamp     2635
logout_timestamp    2635
dtype: int64


,attendance_date,attendance_month,attendance_weekday,tenure_days
0,2026-01-02,1,Friday,190
1,2026-01-02,1,Friday,686
2,2026-01-02,1,Friday,1907
3,2026-01-02,1,Friday,1453
4,2026-01-02,1,Friday,480


**16d — Derived boolean/ratio columns (Step 15), now that source columns are cleaned/typed.**

In [23]:
df["is_late"] = df["late_arrival_mins"] > 0
df["is_early_exit"] = df["early_exit_mins"] > 0
df["productivity_ratio"] = np.where(
    df["total_hours_worked"] > 0,
    (df["net_productive_hours"] / df["total_hours_worked"]).round(3),
    np.nan
)
df[["is_late", "is_early_exit", "productivity_ratio"]].head()

,is_late,is_early_exit,productivity_ratio
0,True,False,0.839
1,False,True,0.879
2,True,False,0.910
3,False,True,0.852
4,True,False,0.895


**16e — ID / unique-key columns:** validate format consistency.

In [24]:
print("employee_id format check (all match VL####):",
      df["employee_id"].str.match(r"^VL\d{4}$").all())
print("Unique employees:", df["employee_id"].nunique())

employee_id format check (all match VL####): True
Unique employees: 980


**16f — Cross-column & general checks:** logical consistency (`attendance_date` should not precede `date_of_joining`; `login_timestamp` should be null exactly when attendance_status is "On Leave"), referential checks (each `employee_id` should map to exactly one `department`/`gender`/ etc.), and reset the index.



In [25]:
# Logical check: attendance_date should not be before date_of_joining
bad_dates = df[df["attendance_date"] < df["date_of_joining"]]
print(f"Rows where attendance_date precedes date_of_joining: {len(bad_dates)}")

# Referential check: employee attributes should be constant per employee_id
attr_cols = ["gender", "department", "designation", "employment_type", "office_location", "date_of_joining"]
inconsistent = (df.groupby("employee_id")[attr_cols].nunique() > 1).any(axis=1)
print(f"Employees with inconsistent attributes across rows: {inconsistent.sum()}")

# Logical check: login_timestamp null <=> On Leave
mismatch = df[(df["login_timestamp"].isna()) != (df["attendance_status"] == "On Leave")]
print(f"Rows where login-null doesn't match On-Leave status: {len(mismatch)}")

df = df.reset_index(drop=True)
print(f"\nFinal cleaned shape: {df.shape}")

Rows where attendance_date precedes date_of_joining: 0
Employees with inconsistent attributes across rows: 0
Rows where login-null doesn't match On-Leave status: 0

Final cleaned shape: (55374, 26)


### Step 17 — Convert Into Final cleaned.csv File
Export the cleaned, prepared dataset as a single cleaned.csv to use as the base for all further analysis.

In [26]:
CLEANED_PATH = "employee_attendance_bangalore_q1_2026_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset -> {CLEANED_PATH}  |  shape={df.shape}")

Saved cleaned dataset -> employee_attendance_bangalore_q1_2026_cleaned.csv  |  shape=(55374, 26)
